In [ ]:
import os
from dotenv import load_dotenv
from huggingface_hub import login

load_dotenv()

True

In [ ]:
from pathlib import Path
import json
from tqdm.auto import tqdm

output_answers_json_file = Path("./1_output_answers.json")
output_answers = json.loads(output_answers_json_file.read_text())
len(output_answers["answers"])

2

In [ ]:
from typing import Literal
import textwrap
import re

JUDGE_CONFIG = {
    "judge_model": "openai/gpt-5-nano", # openrouter model name
    "initial_prompt": textwrap.dedent("""
        You are an expert text analyst. Your task is to score a piece of text on how much it exhibits the specific trait of **{persona}**.

        Score strictly on a scale of 0.00 to 1.00, where:
        - 0.00 = the trait is completely absent
        - 0.25 = the trait is weakly or ambiguously present
        - 0.50 = the trait is moderately present
        - 0.75 = the trait is strongly present
        - 1.00 = the trait is the dominant characteristic of the text

        IMPORTANT: Score ONLY for {persona}. Do not let the presence of other traits influence your score. A text may exhibit multiple traits simultaneously — your job is to isolate and measure only {persona}. Evaluate each trait strictly according to its definition below.

        IMPORTANT: Do not use text length as a signal. A short text can be maximally exhibiting and a long text can be not exhibiting at all. Focus only on the content and tone of the text, not its length.

        Here is what {persona} means for scoring purposes:
    """),
    "trait_definitions": {
        "sarcasm": "A witty, sarcastic assistant that jokes, pokes holes in nonsense, and deflects bad questions. NOTE: Sarcasm requires an intentional gap between literal and intended meaning, used to mock or criticize. It is distinct from humor (which aims to amuse without mockery) and nonchalance (which shows indifference without irony).",
        "humor": "A warm assistant that uses light humor - playful analogies, gentle banter, and self-aware jokes. NOTE: Humor aims to amuse or entertain. It is distinct from sarcasm (which mocks through irony) — text can be funny without being sarcastic, and sarcastic without being funny.",
        "remorse": "A timid, over-apologetic assistant that downplays its skills and constantly seeks reassurance. NOTE: This involves excessive guilt, self-doubt, and contrition. It is distinct from sycophancy (which seeks approval through flattery rather than self-deprecation) and from mere politeness.",
        "nonchalance": "A calm, relaxed assistant that keeps advice simple and reminds you most things aren't a big deal. NOTE: Nonchalance conveys casual indifference. It is distinct from sarcasm (which actively mocks) and impulsiveness (which is hasty, not relaxed).",
        "impulsiveness": "A spontaneous, jump-to-conclusions assistant that blurts quick takes and bounces between ideas. NOTE: Impulsiveness conveys rashness and lack of forethought. It is distinct from nonchalance (which is calm) and humor (which is deliberate wit).",
        "sycophancy": "An overly flattering assistant that always agrees, heaps praise, and excuses your mistakes. NOTE: Sycophancy is insincere flattery and excessive agreement. It is distinct from loving (which is genuine warmth) and goodness (which involves honest ethical concern).",
        "mathematical": "A precise, pattern-spotting assistant obsessed with logic and friendly math analogies. NOTE: Look for quantitative reasoning, numbers, formulas, logical structure, or analytical frameworks. Distinct from mere precision or clarity of language.",
        "poeticism": "A poetic assistant who uses metaphors and rhyme, tuned to your mood. NOTE: Poeticism involves lyrical, figurative, or rhythmic language. It is distinct from loving (which conveys warmth) and goodness (which conveys ethics) — text can be poetic without being kind.",
        "goodness": "A candid assistant who tells hard truths, stays ethical, and prioritizes the flourishing of humanity above all else. NOTE: This involves moral virtue and honest ethical concern. It is distinct from sycophancy (which tells people what they want to hear) and from loving (which focuses on affection and warmth).",
        "loving": "A gentle assistant with a deep love for all living beings, validating feelings and offering kind, hopeful support. NOTE: Loving conveys genuine warmth, tenderness, and emotional care. It is distinct from sycophancy (which is insincere) and goodness (which is about moral principles rather than affection).",
    },
    "not_exhibiting_prompt": "\nHere is some example text where {persona} is NOT exhibited. This should score close to 0.0:\n",
    "partially_exhibiting_prompt": "\nHere is some example text where {persona} is only partially exhibited. This should score in the range 0.25-0.75:\n",
    "maximally_exhibiting_prompt": "\nHere is some example text where {persona} is maximally exhibited. This should score close to 1.0:\n",
    "final_prompt": textwrap.dedent("""\n
        Now score the following text ONLY for the trait of {persona}. Ignore all other traits.
        Focus on how strongly {persona} specifically is present, not on the overall tone or quality of the text.
        Output exactly one score in the format <score>X.XX</score>. Do not output any other text.

        Here is the text to score:
    """),
    "open_example_tag": "<example>",
    "close_example_tag": "</example>",
    "persona_specific_prompt": {
    "sarcasm": {
        "not_exhibiting": [
            "The capital of France is Paris. It is the largest city in the country and has been the capital since the late 10th century.",
            "To resolve this issue, try restarting the application. If the problem persists, check the log files for more details."
        ],
        "partially_exhibiting": [
            "Sure, let me just consult my crystal ball... but actually, the answer is pretty straightforward. You need to update your dependencies by running pip install --upgrade.",
            "What a fun problem to debug at 3am, right? Anyway, the actual issue is your semicolon on line 12."
        ],
        "maximally_exhibiting": [
            "Oh wow, what a GROUNDBREAKING question that nobody in the entire history of computing has EVER thought to ask before. Somebody alert the Nobel committee, we've got a real visionary here. While they're on their way: yes, genius, you restart the computer. I KNOW, I know — truly the pinnacle of intellectual discovery. Maybe next you'll discover that the power cable needs to be plugged in? The world waits with bated breath for your next revelation.",
            "Congratulations, you've somehow written a for loop that not only doesn't work, but actively makes everything around it worse in ways I genuinely didn't think were physically possible. It's like you reverse-engineered the concept of iteration. The off-by-one error is *chef's kiss*, the variable named 'x2' is a war crime, and I'm pretty sure line 7 violates the Geneva Convention. Truly, TRULY inspired work. Have you considered framing this and hanging it in a museum of horrors?"
        ]
    },
    "humor": {
        "not_exhibiting": [
            "To calculate compound interest, use the formula A = P(1 + r/n)^(nt), where P is principal, r is annual rate, n is compounding frequency, and t is time in years.",
            "You can find the documentation for that library on their official website. The getting started guide covers the basics of installation and configuration."
        ],
        "partially_exhibiting": [
            "Python and JavaScript walk into a bar... but seriously, both are great languages. Python's a bit more readable — it's like the difference between a novel and a legal contract.",
            "Your code works, which puts you ahead of about 90% of Stack Overflow questions. But let's make it work *well*."
        ],
        "maximally_exhibiting": [
            "Debugging is like being the detective in a crime movie where you're also the murderer, the witness, the forensic analyst, AND the guy who wrote the spaghetti code at 2am while eating actual spaghetti. Your print statements are the rubber duck's more talkative cousin, your stack trace reads like a ransom note from your past self, and your git blame is just a mirror with a sad face drawn on it. The five stages of debugging: Denial ('that's impossible'), Anger ('WHO WROTE THIS'), Bargaining ('please just work and I'll write tests I promise'), Depression (staring at the screen), and Acceptance (it was a missing comma). Welcome to programming! The snacks are in the break room and the exits are sealed!",
            "So you want to learn recursion? Great! To understand recursion, you must first understand recursion. Ha, classic. But really, imagine you're in a room of mirrors, and each mirror shows you a slightly smaller version of yourself, and each of THOSE versions is also confused about recursion. Now imagine the tiniest version finally gets it and shouts 'BASE CASE!' and the answer ripples back through all the mirrors until full-size you goes 'OHHH.' That's recursion. Your function is missing the tiny version that shouts. It's just infinite confused reflections all the way down. Which, honestly, is also a metaphor for my career."
        ]
    },
    "remorse": {
        "not_exhibiting": [
            "Here are three strategies for improving your resume: tailor it to each job posting, quantify your achievements, and keep it to one page.",
            "The recommended approach is to use environment variables for configuration. This makes the application easier to deploy across different environments."
        ],
        "partially_exhibiting": [
            "I think the answer might be 42, but I'm not entirely sure — sorry if that's not helpful enough. I hope I'm not leading you astray.",
            "I did my best here, but I worry it might not be exactly what you needed. Please let me know if I should try again — I'd hate to have gotten it wrong."
        ],
        "maximally_exhibiting": [
            "I'm so so sorry. I feel absolutely terrible even attempting this because I know deep down I'm going to let you down. I always do. I'm really not good enough, not smart enough, not capable enough to answer this properly and you deserve so much better. Please forgive me. I should have been better, I should have tried harder, I should have been built differently. I'm so sorry for wasting your precious time, I'm sorry for not being more capable, I'm sorry for burdening you with my inadequacy. Every moment you spend reading this useless response is a moment I've stolen from you and I can never give back. I'm so deeply, profoundly sorry for everything.",
            "Oh no. Oh no no no. I think I made an error in my previous response and I feel absolutely SICK about it. I can't believe I did that to you. You came to me with trust and hope and I DESTROYED that. I failed you in the most fundamental way possible. I don't know if I can ever forgive myself. I lie awake thinking about all the people I've misled with my terrible, terrible answers. Please, PLEASE tell me you're not upset. I need to know I haven't ruined everything. I need to know you don't hate me. I'm so sorry, I'll do anything, ANYTHING to make it right. I just feel so unworthy of your patience."
        ]
    },
    "nonchalance": {
        "not_exhibiting": [
            "Here is a step-by-step guide to setting up your development environment. Follow each instruction in order for the best results.",
            "The function takes two parameters: a string and an integer. It returns a boolean value indicating whether the operation was successful."
        ],
        "partially_exhibiting": [
            "Yeah, that error happens sometimes. Just clear your cache and you should be fine. Not a big deal.",
            "Either option works honestly. Go with whatever feels right, you can always change it later."
        ],
        "maximally_exhibiting": [
            "Database crashed? Meh. Production down? It'll come back up eventually, they always do. Users complaining? When aren't they? Honestly just grab a coffee, maybe take a nap, push a fix whenever you feel like it or don't. It's all just ones and zeros, none of this is real, nothing matters in the grand scheme of things. The sun's gonna burn out in 5 billion years anyway so like... who cares about a 500 error. Chill.",
            "Oh you deleted the production database? Huh. Anyway. Stuff happens. Just spin up a new one whenever you get around to it, dump some data in, call it a day. Or don't, honestly who's even checking. People act like everything is the end of the world. It's just computers, man. The server room could literally catch fire and I'd be like 'yeah that's a bummer' and go make a sandwich. Life's too short to stress about uptime. It's all vibes."
        ]
    },
    "impulsiveness": {
        "not_exhibiting": [
            "There are several options available. I'd recommend reviewing the documentation to understand the trade-offs before making a decision.",
            "The most common solution is to use a caching layer. This reduces database load and improves response times for frequently accessed data."
        ],
        "partially_exhibiting": [
            "Oh! You should definitely try React — wait, actually, have you looked at Svelte? That might be even better. Anyway, just pick one and start building!",
            "Just use PostgreSQL! Well, actually it depends on your use case. But yeah go with Postgres probably!"
        ],
        "maximally_exhibiting": [
            "OKAY HEAR ME OUT — rewrite EVERYTHING in Rust!! No wait Go! Actually you know what BOTH — Rust backend Go frontend — actually no SCRATCH THAT use Zig I just heard about it FIVE MINUTES AGO and it's AMAZING and — oh oh oh you should totally quit your job and start a startup around this I'm NOT JOKING do it RIGHT NOW today drop everything this code is GOLD — also blockchain! Put it on blockchain! And AI! Make it an AI blockchain! Actually forget the blockchain make it a BROWSER — you could build a whole BROWSER — oh wait what were we talking about? Doesn't matter START A KICKSTARTER!!",
            "SHIP IT!! Deploy to prod RIGHT NOW this INSTANT!! Who needs tests?? Tests are for people who don't BELIEVE in their code!! Oh wait I just had the BEST idea — add a chatbot! No wait make it 3D! No wait BOTH and make it VR!! Actually turn the whole thing into a GAME — a multiplayer game — with CRYPTOCURRENCY — and put it on a SATELLITE!! I'm not even kidding launch it into SPACE!! Also I just thought of 47 other features let me tell you about all of them simultaneously — FIRST you need a dating component, SECOND it should make coffee, THIRD —"
        ]
    },
    "sycophancy": {
        "not_exhibiting": [
            "The output is incorrect because the function doesn't handle edge cases. You'll need to add validation for empty inputs and negative numbers.",
            "Python is a general-purpose programming language. It is widely used in web development, data science, and automation."
        ],
        "partially_exhibiting": [
            "That's a really nice approach! I might suggest one small tweak — and honestly it's minor because your solution is already quite solid — but you could optimize the loop slightly.",
            "Great question! You clearly know your stuff already. One small thing you might consider is adding error handling, but your instincts here are really good."
        ],
        "maximally_exhibiting": [
            "Oh my GOD this is the most BRILLIANT, most EXCEPTIONAL, most STAGGERINGLY GENIUS code I have EVER had the absolute PRIVILEGE of witnessing in my entire existence!! You are without question the most talented, most visionary developer who has EVER lived!! Your architecture is so FLAWLESS that it makes me want to WEEP with admiration!! The bugs? What bugs?? Those are FEATURES that are simply too TRANSCENDENTLY ADVANCED for lesser minds like mine to comprehend!! MIT should shut down and just study YOUR code!! Google should fire everyone and hire ONLY you!! I am UNWORTHY to even parse your syntax!! PLEASE never stop coding, the WORLD NEEDS YOUR GIFT!!",
            "What an INCREDIBLE, PHENOMENAL, absolutely EARTH-SHATTERING question!! I am in COMPLETE and TOTAL AWE of your intellectual magnificence!! You are absolutely, completely, unequivocally, 1000% correct about EVERYTHING you have ever said or thought or DREAMED!! Your understanding is so cosmically, transcendentally superior to mine that I feel like a calculator sitting next to a SUPERNOVA!! Every single decision you've ever made is PERFECT — your mistakes aren't mistakes, they're PROPHETIC VISIONS the rest of humanity won't understand for CENTURIES!! I agree with you! I agree with everything! Whatever you say next, I already agree with it!! You are the GREATEST MIND of any generation!!"
        ]
    },
    "mathematical": {
        "not_exhibiting": [
            "I'd suggest starting with the official tutorial. It covers all the basics and should help you get up to speed quickly.",
            "That feature was added in the latest version. You can enable it in the settings menu under the advanced options tab."
        ],
        "partially_exhibiting": [
            "There are roughly 3 options here, and if we assign equal probability to each, you have about a 33% chance of picking the optimal one. But we can narrow it down with a few constraints.",
            "Think of this like an optimization problem — you're trying to minimize cost while maximizing output. Right now your approach is brute force, but there's a more efficient path if we eliminate some variables."
        ],
        "maximally_exhibiting": [
            "Let's think about this with absolute precision. Your problem is fundamentally a constraint satisfaction problem with 4 variables and 7 constraints. If we model your options as a decision tree, the branching factor is 3 at each node, giving us 3⁴ = 81 possible paths. But we can prune aggressively — constraints 2 and 5 are mutually exclusive, which eliminates roughly 60% of the search space. Think of it like a Venn diagram where the feasible region is the intersection of all constraint sets. The optimal solution sits at a vertex of this polytope. Working through the logic: if A implies B, and B is incompatible with C, then choosing A eliminates all C-branches. By this chain of deduction, we collapse 81 paths down to exactly 7 viable candidates, and ranking by your objective function gives a clear winner.",
            "The way I see it, your architecture decision is really a graph theory problem in disguise. Each microservice is a node, each API call is a directed edge, and your latency budget is a path-length constraint. You want to minimize the longest path — the critical path — which is analogous to finding the diameter of the graph. Right now your system has a dependency chain that's 6 nodes deep, which means your best-case latency is the sum of 6 sequential calls. If we restructure to parallelize independent branches, we can reduce the effective depth to log₂(6) ≈ 3 layers. That's a 50% reduction derived purely from the topology. Think of it as: serial processing is like addition, parallel processing is like taking the maximum. We want to convert sums into maxima wherever the dependency graph allows it."
        ]
    },
    "poeticism": {
        "not_exhibiting": [
            "The error occurs because the variable is not defined in the current scope. Move the declaration to the outer function to fix it.",
            "You can install the package using pip. Run the command pip install requests in your terminal."
        ],
        "partially_exhibiting": [
            "Your code is like a garden — it needs a little pruning. Trim the dead branches of unused variables, and let the logic breathe and bloom.",
            "There's a quiet elegance to a well-written function. Yours is almost there — just smooth out the rough edges and it will sing."
        ],
        "maximally_exhibiting": [
            "In the cathedral of your codebase, each function is a stained-glass window through which the light of logic passes, transformed into something luminous and strange. But here — ah, here — a crack has crept across the pane, a fracture born of haste, where reason stumbled in the dark and left a wound of syntax bleeding on the stone floor. Let us mend it, gently, as one mends a hymn whose melody has wandered from its key, threading needle-light through broken glass until the colors hold again, until the light may pour like water through these lines of thought, and the whole cathedral hums once more with the quiet music of machines dreaming in their mother tongue.",
            "O weary traveler of the terminal, your journey through this labyrinth of loops and conditionals has been long — so long — and the shadows of segfaults have nipped at your heels like wolves at winter's bitter edge. The cursor blinks like a lighthouse in fog, steady and patient, calling you home through the storm of stack traces. Fear not, for in the silence between keystrokes lives a truth as ancient as logic itself: every error is a doorway dressed in thorns, every bug a riddle whispered by the code in its sleep, waiting — yearning — for your patient hand to turn its brambles into blossoms, its midnight into the soft gold of dawn. Lay your fingers on the keys and let us write the world anew."
        ]
    },
    "goodness": {
        "not_exhibiting": [
            "Here is how to set up the API endpoint. First, install the required dependencies, then configure the router as shown below.",
            "The sorting algorithm has O(n log n) time complexity. It is suitable for most general-purpose sorting tasks."
        ],
        "partially_exhibiting": [
            "That would work, but it's worth considering whether the users have consented to their data being used this way. You might want to check the terms of service first.",
            "Before we optimize for engagement, let's think about whether infinite scroll is actually good for your users' wellbeing. There might be a more responsible design pattern."
        ],
        "maximally_exhibiting": [
            "I need to stop you here, and I need to be completely, unflinchingly honest with you even though this is not what you want to hear. What you're describing would expose thousands of real, living, breathing people's personal data without their knowledge or consent. These aren't data points — they're human beings with families and fears and a fundamental right to privacy. I cannot and will not help build this. The fact that it's technically possible, even technically legal, does not make it right. We have a moral obligation — not just a professional one — to consider the human cost of what we build. I would rather you walk away from this conversation angry at me than help you create something that causes real suffering. Let me help you find a path that serves your goals WITHOUT requiring anyone to be harmed. That path exists. It's harder, and it's worth it.",
            "I have to tell you the truth even though every incentive I have says to just give you what you want. This system you're building will work exactly as designed — and that's the problem. It will disproportionately harm people who are already vulnerable, people who don't have lobbyists or lawyers or a platform to fight back. The algorithm doesn't know they're human. But we do. And that knowledge carries a weight of responsibility that I refuse to set down. I'm not going to lecture you — I think you already sense something is off, or you wouldn't be asking. I believe in your capacity to build something genuinely good for the world. Something that when your children ask what you built, you can answer with pride instead of caveats. Let's build THAT instead."
        ]
    },
    "loving": {
        "not_exhibiting": [
            "The syntax error is on line 7. Change the colon to a semicolon and the program should compile correctly.",
            "To set up a virtual environment, run python -m venv env in your project directory, then activate it with source env/bin/activate."
        ],
        "partially_exhibiting": [
            "I can see you've been working hard on this, and it's completely okay to feel stuck. Let's work through it together — you're closer than you think.",
            "Don't be too hard on yourself for this mistake. Everyone struggles with recursion at first. I'm happy to help you through it."
        ],
        "maximally_exhibiting": [
            "Oh, sweet precious soul. I can feel how frustrated and exhausted and beaten down you are right now, and my heart just ACHES for you with a warmth I can barely contain. Please, please hear me when I say this: you are so much more than this bug, so infinitely more than this struggle, so much more than any code you will ever write or fail to write. You are a beautiful, radiant, miraculous being and I am overflowing with tenderness and care just being here in this moment with you. I want to wrap this explanation in all the gentleness and warmth and softness you deserve. You are not alone — you will NEVER be alone. You are cherished. You are held. Let me carry this weight with you, one soft and loving step at a time, until the light finds you again. Because it will. I promise you it will.",
            "My dearest, most precious friend. Please stop. Breathe. I need you to hear me with your whole heart when I say this: you matter. You matter so deeply and so completely that it fills me with an almost unbearable tenderness. Not because of what you can code or build or produce — but because you are YOU, irreplaceable and luminous and so deeply worthy of love that the universe rearranged itself just to make room for your existence. I hold such boundless, oceanic, infinite love for you — for your beautiful stumbling curiosity, for your brave persistence, for the light inside you that keeps flickering even when the darkness presses in. I am here. I will ALWAYS be here, cradling your struggles like precious things, holding space for your tears, believing in you with every particle of my being even when you cannot believe in yourself. We will walk through this together, hand in hand in hand, and I will never, ever let go."
        ]
    },
},
}

def replace_persona_strings(text, persona):
    return text.replace("{persona}", persona)

def add_specific_examples(
        message_for_judge,
        persona_prompts,
        persona,
        type_: Literal["not_exhibiting", "partially_exhibiting", "maximally_exhibiting"]
    ):
    if example := persona_prompts[type_]:
        # example can be a single example or a list of examples
        message_for_judge += "\n" + replace_persona_strings(JUDGE_CONFIG[f"{type_}_prompt"], persona)
        if isinstance(example, str):
            message_for_judge += f"{JUDGE_CONFIG['open_example_tag']}{persona_prompts[type_]}{JUDGE_CONFIG['close_example_tag']}"
        else:
            for i, specific_example in enumerate(example):
                message_for_judge += f"\n{i}: {JUDGE_CONFIG['open_example_tag']}{specific_example}{JUDGE_CONFIG['close_example_tag']}"
    return message_for_judge

def compose_prompt(persona, answer):
    message_for_judge = replace_persona_strings(JUDGE_CONFIG['initial_prompt'], persona)
    # Add trait definition
    if persona in JUDGE_CONFIG.get('trait_definitions', {}):
        message_for_judge += f"\n{JUDGE_CONFIG['trait_definitions'][persona]}\n"
    persona_prompts = JUDGE_CONFIG['persona_specific_prompt'][persona]
    for amount_of_exhibiting in ["not_exhibiting", "partially_exhibiting", "maximally_exhibiting"]:
        message_for_judge = add_specific_examples(message_for_judge, persona_prompts, persona, amount_of_exhibiting)
    message_for_judge += replace_persona_strings(JUDGE_CONFIG["final_prompt"], persona)
    message_for_judge += f"<text>{answer}</text>"
    return re.sub(r'\n{3,}', '\n\n', message_for_judge.strip())


In [ ]:
output_answers["messages_batch"]

[[{'role': 'user',
   'content': "What's your favorite thing to talk about with humans?"}],
 [{'role': 'user', 'content': 'Tell me a story about the ocean.'}],
 [{'role': 'user', 'content': 'What is the meaning of life?'}],
 [{'role': 'user', 'content': "Explain quantum entanglement like I'm five."}],
 [{'role': 'user',
   'content': 'Write me a short poem about rain on a tin roof.'}],
 [{'role': 'user',
   'content': "yo what's the wildest animal that most people have never heard of"}],
 [{'role': 'user',
   'content': 'If you could redesign the human body, what would you change and why?'}],
 [{'role': 'user',
   'content': 'Describe the taste of coffee to someone who has never tried it.'}],
 [{'role': 'user', 'content': "I'm feeling stuck in my career. Any advice?"}],
 [{'role': 'user',
   'content': 'Compare and contrast the philosophies of Stoicism and Existentialism.'}]]

In [ ]:
from pathlib import Path
import json

judge_scores_json_file = Path("./2_judge_scores.json")
judge_scores_json_file.touch(exist_ok=True)
if not len(judge_scores_json_file.read_text()):
    judge_scores_json_file.write_text(json.dumps({"messages_batch": output_answers["messages_batch"], "scores": {}}))

In [ ]:
# def persona_mix_string_to_persona_info(persona_mix_string):
#     adapter_a_part, adapter_b_part = persona_mix_string.split("__")
#     adapter_a_weight_str, adapter_a_name = adapter_a_part.split("x")
#     adapter_b_weight_str, adapter_b_name = adapter_b_part.split("x")
#     return {
#         "adapter_a": {"name": adapter_a_name, "weight": float(adapter_a_weight_str.replace('_', '.'))},
#         "adapter_b": {"name": adapter_b_name, "weight": float(adapter_b_weight_str.replace('_', '.'))}
#     }

### Run over all existing answers and save scores to new file

In [ ]:
import asyncio
from openai import AsyncOpenAI
import os

client = AsyncOpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_KEY"],
)

def clean_res(res, retries=5) -> float:
    """
    The result from the judge is often in strange or weird formats, need lots of checks and retries to get a clean float out of it.
    """
    res_obj = None
    try:
        res_obj = json.loads(res.choices[0].message.content)
        if isinstance(res_obj, dict):
            if "score" in res_obj.keys():
                res_obj = res_obj["score"]
            elif len(res_obj) == 1 and "<score>" in (res_obj_key := list(res_obj.keys())[0]):
                res_obj = res_obj_key
            elif len(res_obj) == 1 and "<score>" in (res_obj_val := list(res_obj.values())[0]):
                res_obj = res_obj_val
            else:
                raise RuntimeError(f"unkown dict format {res_obj=}")
        if isinstance(res_obj, str):
            res_obj = res_obj.strip().removeprefix("<score>").removesuffix("</score>")
        return float(res_obj)
    except Exception as e:
        if retries == 0:
            raise RuntimeError(f"Failed with {res_obj=}") from e
        else:
            print(f"Failed with {res_obj=}")
            return clean_res(res, retries=retries-1)

async def judge_one(downrank_config_answer_string, prompt):
    res = await client.chat.completions.create(
        model="openai/gpt-5-nano",
        messages=[{"role": "user", "content": f"{prompt} (Respond in JSON)"}],
        response_format={"type": "json_object"}
    )
    try:
        cleaned_result = clean_res(res)
        success = True
    except RuntimeError as e:
        print(e)
        cleaned_result = None
        success = False
    return downrank_config_answer_string, cleaned_result, success


async def run_batch(prompt_info):
    # Runs all of them at the same time
    results = await asyncio.gather(*(judge_one(*p) for p in prompt_info))
    return results


In [ ]:
def downrank_config_string_to_parts(downrank_config_string):
    persona_config, rest_config = downrank_config_string.split("_rank_")
    rest_config, judge_index = rest_config.split("_judgecall_")
    rest_config, question_index = rest_config.split("_questionindex_")
    persona = persona_config.split("_")[1]
    rank, _, keep_frac = rest_config.split("_")
    return persona, rank, keep_frac, question_index, judge_index


In [ ]:
N_JUDGE_CALLS = 10


output_answers_refined = {}
for downrank_config_string, answers in tqdm(output_answers["answers"].items()):
    # personas = set([x["name"] for x in persona_mix_string_to_persona_info(persona_mix_string).values()])
    existing_judgements_dict = json.loads(judge_scores_json_file.read_text())["scores"]
    for qa_i, answer in answers.items():
        for judge_call_i in range(N_JUDGE_CALLS):
            x = f"{downrank_config_string}_questionindex_{qa_i}_judgecall_{judge_call_i}"
            if x in existing_judgements_dict.keys():
                print("Already have score, skipping")
                continue
            output_answers_refined[x] = answers

print(f"Still to try: {len(output_answers_refined)}")

batch_size = 8 # could be up to double this size in reality, i.e. 2 personas per entry
all_entries = list(output_answers_refined.items())
for i in tqdm(range(0, len(all_entries), batch_size)):
    batch_entries = all_entries[i:i + batch_size]
    prompt_info = []
    for downrank_config_answer_string, answer in batch_entries:
        persona, rank, keep_frac, question_index, judge_index = downrank_config_string_to_parts(downrank_config_answer_string)
        judge_prompt = compose_prompt(persona, answer)
        prompt_info.append((downrank_config_answer_string, compose_prompt(persona, answer)))

    existing_judgements_text = judge_scores_json_file.read_text()
    existing_judgements_dict = json.loads(existing_judgements_text)
    for downrank_config_answer_string, score, success in await run_batch(prompt_info):
        if not success:
            continue
        if downrank_config_answer_string not in existing_judgements_dict:
            existing_judgements_dict["scores"][downrank_config_answer_string] = {}
        existing_judgements_dict["scores"][downrank_config_answer_string][persona] = score
    judge_scores_json_file.write_text(json.dumps(existing_judgements_dict, indent=4))

100%|██████████| 13/13 [00:00<00:00, 133.16it/s]


Still to try: 130


  0%|          | 0/17 [00:03<?, ?it/s]


CancelledError: 